# Lab — Cosine Similarity

**Objective:** Compute cosine similarity from first principles and rank paraphrase candidates.

**Book track:** `03-language-and-representation` · **Time:** 30–45 minutes · **Python:** 3.10+

Work through the cells in order. Each code cell should run top-to-bottom. Keep `main.py` in this folder aligned with your final answers—`pytest` validates that file.


## How to use this notebook

1. Open from the lab directory (`labs/01-cosine-similarity/`) in Jupyter, VS Code, or Codespaces.
2. Run cells sequentially; restart the kernel if you change earlier definitions.
3. Complete **Your turn** sections, then sync working code into `main.py`.
4. Run the verification cell (`pytest`) before you finish.


In [ ]:
from pathlib import Path

LAB_DIR = Path('.').resolve()
assert (LAB_DIR / 'main.py').exists(), (
    'Start Jupyter from the lab directory, e.g. labs/01-cosine-similarity/'
)
print('Lab directory:', LAB_DIR)


## Tasks

1. Predict ranked output before running `main.py`.
2. Add orthogonal and zero-vector cases to `test_lab.py`.
3. Compare cosine vs dot product on unnormalized vectors.
4. Document when magnitude should matter for your retrieval task.


## Step 1 — Vectors, dot product, and magnitude

Cosine similarity measures the angle between two vectors. It ignores magnitude after normalization—useful when comparing embedding directions.


In [ ]:
from math import sqrt


def dot(a: list[float], b: list[float]) -> float:
    if len(a) != len(b):
        raise ValueError('vectors must have equal dimensions')
    return sum(x * y for x, y in zip(a, b))


def magnitude(v: list[float]) -> float:
    return sqrt(sum(x * x for x in v))


query = [1.0, 1.0, 0.0]
candidate = [0.9, 1.0, 0.1]
print('dot product:', dot(query, candidate))
print('magnitudes:', magnitude(query), magnitude(candidate))


## Step 2 — Cosine similarity

Implement cosine and handle edge cases (zero vectors, mismatched lengths).


In [ ]:
def cosine(a: list[float], b: list[float]) -> float:
    if len(a) != len(b):
        raise ValueError('vectors must have equal dimensions')
    dot_ab = dot(a, b)
    norm_a = magnitude(a)
    norm_b = magnitude(b)
    if norm_a == 0 or norm_b == 0:
        raise ValueError('cosine similarity is undefined for a zero vector')
    return dot_ab / (norm_a * norm_b)


print('cosine(query, candidate):', round(cosine(query, candidate), 3))


## Step 3 — Rank paraphrase candidates

Rank short text labels by similarity to a query vector.


In [ ]:
candidates = {
    'service unavailable': [0.9, 1.0, 0.1],
    'invoice approved': [0.1, 0.0, 1.0],
    'application outage': [1.0, 0.8, 0.0],
}

ranked = sorted(
    ((cosine(query, vector), text) for text, vector in candidates.items()),
    reverse=True,
)

print('Ranked candidates:')
for score, text in ranked:
    print(f'  {score:.3f}  {text}')


## Your turn

1. **Predict** the ranked order before running Step 3.
2. Compare **cosine** vs raw **dot product** on unnormalized vectors.
3. Try orthogonal vectors and a zero vector—what should happen?
4. Copy the final `cosine` function (and ranking loop) into `main.py`.


In [ ]:
# Example: cosine vs dot product on different magnitudes
short = [1.0, 0.0]
long = [10.0, 0.0]
print('dot:', dot(short, long))
print('cosine:', cosine(short, long))

# TODO: add orthogonal and zero-vector checks, then update main.py


## Verify

Run the test suite against `main.py` and `test_lab.py`.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'test_lab.py', '-q'],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
assert result.returncode == 0, 'Tests failed—see output above'


## Reflection

- What broke first when you changed inputs?
- Which simpler baseline would you compare against in a design review?

## Extensions

- Add another case to `test_lab.py`.
- Link observations to a concept card on the AIEBOK site.
